# Sequence Tokenization

Base utilizada: [ParaCrawl](https://lindat.mff.cuni.cz/repository/items/413b6bdb-e898-4297-b7c6-aac14dccde64)

In [138]:
import torch
import torch.nn as nn
import numpy as np

In [66]:
enFile = "./paracrawl-release1.en-pt.zipporah0-dedup-clean"
ptFile = "./paracrawl-release1.en-pt.zipporah0-dedup-clean.pt"

In [67]:
START_TOKEN = "<START>"
PADDING_TOKEN = "<PADDING>"
END_TOKEN = "<END>"

enVocabulary = [START_TOKEN, ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', 
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
    ':', '<', '=', '>', '?', '@', 
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 
    'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 
    'Y', 'Z',
    '[', '\\', ']', '^', '_', '`', 
    'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l',
    'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 
    'y', 'z', 
    '{', '|', '}', '~', PADDING_TOKEN, END_TOKEN]

ptVocabulary = [START_TOKEN, ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', 
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
    ':', '<', '=', '>', '?', '@', 
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 
    'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 
    'Y', 'Z',
    'Ã', 'Á', 'À', 'Â', 'É', 'Ê', 'Í', 'Ó', 'Ô', 'Õ', 'Ú', 'Ç',
    '[', '\\', ']', '^', '_', '`', 
    'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l',
    'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 
    'y', 'z',
    'ã', 'á', 'à', 'â', 'é', 'ê', 'í', 'ó', 'ô', 'õ', 'ú', 'ç',
    '{', '|', '}', '~', PADDING_TOKEN, END_TOKEN]

In [92]:
# Criando hash maps para acessar vocabulário
idxToEn = {k:v for k,v in enumerate(enVocabulary)}
enToIdx = {v:k for k,v in enumerate(enVocabulary)}
idxToPt = {k:v for k,v in enumerate(ptVocabulary)}
ptToIdx = {v:k for k,v in enumerate(ptVocabulary)}

In [93]:
# Lê arquivos
with open(enFile, 'r') as file:
    enSentences = file.readlines()
with open(ptFile, 'r') as file:
    ptSentences = file.readlines()

# Limitar número de sentenças
TOTAL_SENTENCES = 100000
enSentences = enSentences[:TOTAL_SENTENCES] # Limita
ptSentences = ptSentences[:TOTAL_SENTENCES] # Limita
enSentences = [sentence.rstrip('\n') for sentence in enSentences] # Tira '\n'
ptSentences = [sentence.rstrip('\n') for sentence in ptSentences] # Tira '\n'

In [94]:
enSentences[:10]

['More and more businesses are changing their marketing campaigns to heavily rely on, if not fully incorporate, the use of social media to promote their company and its products.',
 'But, promotion is only a small drop in the waterfall of SMM strategies.',
 'The real success of social media marketing is not necessarily profitability.',
 'The primary reasons why SMM is important is because it allows brands to engage with an online community about the company, its products and its services.',
 'In other words, creating a community around the brand is the goal.',
 'Through social media, a brand can become a trusted friend and gain a loyal following.',
 'Because community creation is the goal, brand visibility should take center stage.',
 'Focusing on these SMM strategies provides our SMM experts a platform to fully assess and delve into the following issues when looking at a client’s social media presence and make recommendations accordingly:',
 'Trust management: Social media has vast di

In [95]:
ptSentences[:10]

['Mais e mais empresas estão mudando suas campanhas de marketing para dependem grandemente, se não incorporar plenamente, o uso das mídias sociais para promover a sua empresa e seus produtos.',
 'Mas, a promoção é apenas uma pequena queda na cachoeira de estratégias de SMM.',
 'O verdadeiro sucesso de marketing de mídia social não é necessariamente a rentabilidade.',
 'As principais razões pelas quais SMM é importante porque permite que as marcas se envolver com uma comunidade online sobre a empresa, seus produtos e seus serviços.',
 'Em outras palavras, criar uma comunidade em torno da marca é a meta.',
 'Através de meios de comunicação social, uma marca pode se tornar um amigo de confiança e ganhar um público fiel.',
 'Porque a criação de comunidade é o objetivo, a visibilidade da marca deve tomar o centro do palco.',
 'Focalizando essas estratégias SMM fornece aos nossos especialistas SMM uma plataforma para avaliar plenamente e aprofundar as seguintes questões quando se olha para a

In [96]:
max(len(x) for x in enSentences), max(len(x) for x in ptSentences)

(1086, 1161)

In [97]:
# O 97° percentil significa que este valor está acima de 97% dos valores no teste.
# Utilizamos para obter max_seq_len
PERCENTILE = 97

print( f"{PERCENTILE}th percentile length English: {np.percentile([len(x) for x in enSentences], PERCENTILE)}" )
print( f"{PERCENTILE}th percentile length Portuguese: {np.percentile([len(x) for x in ptSentences], PERCENTILE)}" )

97th percentile length English: 346.0
97th percentile length Portuguese: 368.0


In [ ]:
max_seq_len = 400

# Iremos filtrar as sentenças válidas (tratar dataset)
def is_valid_tokens(sentence, vocab):
    for token in sentence:
        if token not in vocab:
            return False
    return True

def is_valid_length(sentence, max_seq_len):
    return len(list(sentence)) < (max_seq_len - 1)

valid_sentence_idx = []

for i in range(len(ptSentences)):
    ptSentence, enSentence = ptSentences[i], enSentences[i]
    if is_valid_length(ptSentence, max_seq_len) \
        and is_valid_length(enSentence, max_seq_len) \
        and is_valid_tokens(ptSentence, ptVocabulary) \
        and is_valid_tokens(enSentence, enVocabulary):
            valid_sentence_idx.append(i)

print(f"Number of sentences: {len(ptSentences)}")
print(f"Number of valid sentences: {len(valid_sentence_idx)}")

[]
Number of sentences: 100000
Number of valid sentences: 80006


In [107]:
from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, enSentences, ptSentences):
        self.enSentences = enSentences
        self.ptSentences = ptSentences

    # len(dataset) retorna isso aqui
    def __len__(self):
        return len(self.enSentences)

    # dataset[i] vai retornar isso aqui
    def __getitem__(self, idx):
        return self.enSentences[idx], self.ptSentences[idx]

enFilteredSentences = [enSentences[i] for i in valid_sentence_idx]
ptFilteredSentences = [ptSentences[i] for i in valid_sentence_idx]

dataset = TextDataset(enFilteredSentences, ptFilteredSentences)

In [108]:
len(dataset)

80006

In [109]:
dataset[1]

('But, promotion is only a small drop in the waterfall of SMM strategies.',
 'Mas, a promoção é apenas uma pequena queda na cachoeira de estratégias de SMM.')

In [110]:
batch_size = 3
train_loader = DataLoader(dataset, batch_size)
iterator = iter(train_loader)

for batch_num, batch in enumerate(iterator):
    print(batch)
    if batch_num > 3:
        break

[('More and more businesses are changing their marketing campaigns to heavily rely on, if not fully incorporate, the use of social media to promote their company and its products.', 'But, promotion is only a small drop in the waterfall of SMM strategies.', 'The real success of social media marketing is not necessarily profitability.'), ('Mais e mais empresas estão mudando suas campanhas de marketing para dependem grandemente, se não incorporar plenamente, o uso das mídias sociais para promover a sua empresa e seus produtos.', 'Mas, a promoção é apenas uma pequena queda na cachoeira de estratégias de SMM.', 'O verdadeiro sucesso de marketing de mídia social não é necessariamente a rentabilidade.')]
[('The primary reasons why SMM is important is because it allows brands to engage with an online community about the company, its products and its services.', 'In other words, creating a community around the brand is the goal.', 'Through social media, a brand can become a trusted friend and g

In [111]:
def tokenize(sentence, langToIdx, startToken=True, endToken=True):
    # Insere índice dos tokens
    sentenceWordIdx = [langToIdx[token] for token in list(sentence)]
    # Coloca <START> no início
    if startToken:
        sentenceWordIdx.insert(0, langToIdx[START_TOKEN])
    # Coloca <END> no final
    if endToken:
        sentenceWordIdx.append(langToIdx[END_TOKEN])
    # No resto, coloca <PADDING>
    for _ in range(len(sentenceWordIdx), max_seq_len):
        sentenceWordIdx.append(langToIdx[PADDING_TOKEN])
    return torch.tensor(sentenceWordIdx)

Os tokens aqui serão simplesmente os índices dos símbolos! (a -> 3)

In [112]:
batch

[('These operators have a particular meaning to each of the different search engines, but not all engines accept the same operators.',
  'The return provides entirely different results than the average search.',
  'In this particular case, the operator used is link: followed by the domain name.'),
 ('Estes operadores têm um significado particular para cada um dos motores de busca diferentes, mas nem todos os motores de aceitar os mesmos operadores.',
  'O retorno fornece resultados completamente diferentes do que a procura média.',
  'Neste caso particular, o operador usado é o link: seguido do nome de domínio.')]

In [132]:
enTokenized, ptTokenized = [], []

for sentenceNum in range(batch_size):
    enSentence, ptSentence = batch[0][sentenceNum], batch[1][sentenceNum]
    enTokenized.append(tokenize(enSentence, enToIdx, startToken=False, endToken=False))
    ptTokenized.append(tokenize(ptSentence, ptToIdx, startToken=True, endToken=True))

# Antes: enTokenized = [tensor([]), tensor[], tensor[]] -> Lista de tensores
enTokenized = torch.stack(enTokenized)
# Depois: enTokenized = tensor([[],[],[]]) -> Um tensor só
ptTokenized = torch.stack(ptTokenized)

In [133]:
# Consulta
print(enToIdx['T'], enToIdx['h'], enToIdx['e'], enToIdx['y'], end=" ")

enTokenized

52 72 69 89 

tensor([[52, 72, 69,  ..., 95, 95, 95],
        [52, 72, 69,  ..., 95, 95, 95],
        [41, 78,  1,  ..., 95, 95, 95]])

In [ ]:
NEG_INFTY = -1e9 # Por causa da softmax

def create_masks(enBatch, ptBatch):
    nSentences = len(enBatch)

    lookAheadMask = torch.full([max_seq_len, max_seq_len], True)
    lookAheadMask = torch.triu(lookAheadMask, diagonal=1)

    encoderPaddingMask = torch.full([nSentences, max_seq_len, max_seq_len], False)
    decoderPaddingMaskSelfAttention = torch.full([nSentences, max_seq_len, max_seq_len], False)
    decoderPaddingMaskCrossAttention = torch.full([nSentences, max_seq_len, max_seq_len], False)

    for idx in range(nSentences):
        enSentenceLength, ptSentenceLength = len(enBatch[idx]), len(ptBatch[idx])

        enCharsToPaddingMask = np.arange(enSentenceLength + 1, max_seq_len)
        ptCharsToPaddingMask = np.arange(ptSentenceLength + 1, max_seq_len)

        encoderPaddingMask[idx, :, enCharsToPaddingMask] = True
        encoderPaddingMask[idx, enCharsToPaddingMask, :] = True

        decoderPaddingMaskSelfAttention[idx, :, ptCharsToPaddingMask] = True
        decoderPaddingMaskSelfAttention[idx, ptCharsToPaddingMask, :] = True

        decoderPaddingMaskCrossAttention[idx, :, enCharsToPaddingMask] = True
        decoderPaddingMaskCrossAttention[idx, ptCharsToPaddingMask, :] = True

    encoderSelfAttentionMask = torch.where(encoderPaddingMask, NEG_INFTY, 0)
    decoderSelfAttentionMask = torch.where(lookAheadMask + decoderPaddingMaskSelfAttention, NEG_INFTY, 0)
    decoderCrossAttentionMask = torch.where(decoderPaddingMaskCrossAttention, NEG_INFTY, 0)

    print(f"encoderSelfAttentionMask {encoderSelfAttentionMask.size()}: {encoderSelfAttentionMask[0, :10, :10]}")
    print(f"decoderSelfAttentionMask {decoderSelfAttentionMask.size()}: {decoderSelfAttentionMask[0, :10, :10]}")
    print(f"decoderCrossAttentionMask {decoderCrossAttentionMask.size()}: {decoderCrossAttentionMask[0, :10, :10]}")

    return encoderSelfAttentionMask, decoderSelfAttentionMask, decoderCrossAttentionMask

# DecoderSelfAttentionMask: matriz triangular inferior cheia de zeros (em cima, infinitos negativos)
create_masks(batch[0], batch[1])


encoderSelfAttentionMask torch.Size([3, 400, 400]): tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])
decoderSelfAttentionMask torch.Size([3, 400, 400]): tensor([[ 0.0000e+00, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09,
         -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09],
        [ 0.0000e+00,  0.0000e+00, -1.0000e+09, -1.0000e+09, -1.0000e+09,
         -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00, -1.0000e+09, -1.0000e+09,
         -1.0000e+09, -

(tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          ...,
          [-1.0000e+09, -1.0000e+09, -1.0000e+09,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          [-1.0000e+09, -1.0000e+09, -1.0000e+09,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          [-1.0000e+09, -1.0000e+09, -1.0000e+09,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09]],
 
         [[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ..., -1.0000e+09,
           -1.0000e+09, -1.0000e+09],
          ...,
    

In [ ]:
class SentenceEmbedding(nn.Module):
    "For a given sentence, create an embedding"
    def __init__(self, maxSeqLen, dModel, langToIdx, START_TOKEN, END_TOKEN, PADDING_TOKEN):
        super().__init__()
        self.vocabSize = len(langToIdx)
        self.maxSeqLen = maxSeqLen
        self.embedding = nn.Embedding(self.vocabSize, dModel)
        self.langToIdx = langToIdx
        self.positionEncoder = PositionalEncoding(dModel, maxSeqLen)
        self.dropout = nn.Dropout(p=0.1)
        self.START_TOKEN = START_TOKEN
        self.END_TOKEN = END_TOKEN
        self.PADDING_TOKEN = PADDING_TOKEN
        self.device = torch.device(
            "cuda" if torch.cuda.is_available()
            else "cpu"
        )

    def batch_tokenize(self, batch, startToken=True, endToken=True):
        def tokenize(sentence, startToken=True, endToken=True):
            sentenceWordIdx = [self.langToIdx[token] for token in list(sentence)]
            if startToken:
                sentenceWordIdx.insert(0, self.langToIdx[self.START_TOKEN])
            if endToken:
                sentenceWordIdx.append(self.langToIdx[self.END_TOKEN])
            for _ in range(len(sentenceWordIdx), self.maxSeqLen):
                sentenceWordIdx.append(self.langToIdx[self.PADDING_TOKEN])

            return torch.tensor(sentenceWordIdx)

        tokenized = []
        for sentenceNum in range(len(batch)):
            tokenized.append(tokenize(batch[sentenceNum], startToken, endToken))
        tokenized = torch.stack(tokenized)
        return tokenized.to(self.device)

    def forward(self, x, endToken=True):
        x = self.batch_tokenize(x, endToken)
        x = self.embedding(x)
        pos = self.positionEncoder().to(self.device)
        x = self.dropout(x + pos)
        return x